In [1]:
"""
Global Superstore - Phase 2 | Notebook 02
Discount Impact Analysis
File: 02_Python/02_Discount_Analysis.py

Chay: python 02_Python/02_Discount_Analysis.py
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import sqlite3
import os
import warnings
warnings.filterwarnings("ignore")

# ── Setup ─────────────────────────────────────────────────────────────────────
DB_FILE   = r"C:\DA\global-superstore-analysis\data\superstore.db"
CHART_DIR = r"C:\DA\global-superstore-analysis\02_Python\charts"
os.makedirs(CHART_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.size":        11,
})

BLUE   = "#2E75B6"
RED    = "#C0392B"
ORANGE = "#E67E22"
GREEN  = "#27AE60"
GRAY   = "#95A5A6"

print("=" * 60)
print("GLOBAL SUPERSTORE - DISCOUNT IMPACT ANALYSIS")
print("=" * 60)

# ── Load data ──────────────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_FILE)
df   = pd.read_sql("SELECT * FROM orders", conn)
conn.close()

print(f"\nData loaded: {df.shape[0]:,} rows")

# Tao cot discount bracket
def disc_bracket(d):
    if d == 0:      return "0% (No disc)"
    elif d <= 0.10: return "1-10%"
    elif d <= 0.20: return "11-20%"
    elif d <= 0.30: return "21-30%"
    elif d <= 0.40: return "31-40%"
    else:           return ">40%"

ORDER = ["0% (No disc)", "1-10%", "11-20%", "21-30%", "31-40%", ">40%"]
df["disc_bracket"] = df["discount"].apply(disc_bracket)
df["disc_bracket"] = pd.Categorical(df["disc_bracket"], categories=ORDER, ordered=True)
df["is_loss"]      = df["profit"] < 0

# ── SUMMARY STATS ─────────────────────────────────────────────────────────────
print("\n── DISCOUNT IMPACT SUMMARY ──────────────────────────────────")
summary = df.groupby("disc_bracket", observed=True).agg(
    orders      = ("order_id",  "count"),
    total_sales = ("sales",     "sum"),
    total_profit= ("profit",    "sum"),
    loss_orders = ("is_loss",   "sum"),
).reset_index()
summary["margin_pct"]    = summary["total_profit"] / summary["total_sales"] * 100
summary["loss_rate_pct"] = summary["loss_orders"]  / summary["orders"]       * 100
print(summary.to_string(index=False))

# ── CHART 7: Profit Margin theo Discount Bracket ─────────────────────────────
print("\n[Chart 7] Profit Margin by Discount Bracket...")

bar_colors = [GREEN if m > 0 else RED for m in summary["margin_pct"]]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(summary["disc_bracket"], summary["margin_pct"],
              color=bar_colors, width=0.55, edgecolor="white", linewidth=0.8)
ax.axhline(0, color="black", linewidth=1.2, linestyle="--")

# Them nhan tren moi cot
for bar, val, orders in zip(bars, summary["margin_pct"], summary["orders"]):
    y = bar.get_height()
    offset = 0.5 if y >= 0 else -1.5
    ax.text(bar.get_x() + bar.get_width()/2,
            y + offset,
            f"{val:.1f}%\n({orders:,} orders)",
            ha="center", va="bottom" if y >= 0 else "top",
            fontsize=9, fontweight="bold")

# Vung thua lo
ax.axvspan(3.5, 5.5, alpha=0.08, color=RED, label="Loss zone (Disc > 30%)")

ax.set_xlabel("Discount Range")
ax.set_ylabel("Profit Margin (%)")
ax.set_title("Chart 7 — Profit Margin by Discount Range\n"
             "Key Finding: Discount > 20% consistently leads to losses",
             fontsize=13, fontweight="bold", pad=12)

green_p = mpatches.Patch(color=GREEN, label="Profitable")
red_p   = mpatches.Patch(color=RED,   label="Loss-making")
ax.legend(handles=[green_p, red_p], loc="lower left")
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_07_discount_margin.png"), dpi=150)
plt.close()
print("   Saved: chart_07_discount_margin.png")

# ── CHART 8: Loss Rate by Discount Bracket ────────────────────────────────────
print("[Chart 8] Loss Rate by Discount Bracket...")

fig, ax = plt.subplots(figsize=(9, 5))
bars2 = ax.bar(summary["disc_bracket"], summary["loss_rate_pct"],
               color=[GREEN if r < 30 else ORANGE if r < 70 else RED
                      for r in summary["loss_rate_pct"]],
               width=0.55, edgecolor="white")

ax.axhline(50, color="red", linewidth=1.2, linestyle="--",
           label="50% loss threshold")

for bar, val in zip(bars2, summary["loss_rate_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1,
            f"{val:.0f}%",
            ha="center", fontsize=10, fontweight="bold")

ax.set_ylim(0, 110)
ax.set_xlabel("Discount Range")
ax.set_ylabel("% Orders with Negative Profit")
ax.set_title("Chart 8 — Loss Order Rate by Discount Range\n"
             "Key Finding: >40% discount = 100% of orders are losses",
             fontsize=13, fontweight="bold", pad=12)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_08_loss_rate.png"), dpi=150)
plt.close()
print("   Saved: chart_08_loss_rate.png")

# ── CHART 9: Discount vs Profit by Category ───────────────────────────────────
print("[Chart 9] Discount vs Profit by Category...")

CATEGORY_COLORS = {
    "Furniture":        "#E74C3C",
    "Technology":       "#2E75B6",
    "Office Supplies":  "#27AE60",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, cat in zip(axes, ["Technology", "Furniture", "Office Supplies"]):
    data = df[df["category"] == cat].sample(min(2000, len(df[df["category"]==cat])),
                                             random_state=42)
    c = [CATEGORY_COLORS[cat] if p >= 0 else RED for p in data["profit"]]
    ax.scatter(data["discount"]*100, data["profit"],
               c=c, alpha=0.3, s=12)
    ax.axhline(0, color="black", linewidth=1, linestyle="--")
    ax.axvline(20, color=ORANGE, linewidth=1.2, linestyle=":", label="20% disc")

    # Trend line
    z = np.polyfit(data["discount"]*100, data["profit"], 1)
    x_l = np.linspace(0, 90, 100)
    ax.plot(x_l, np.poly1d(z)(x_l), color="navy", linewidth=2)

    margin = df[df["category"]==cat]["profit"].sum() / df[df["category"]==cat]["sales"].sum() * 100
    ax.set_title(f"{cat}\n(Margin: {margin:.1f}%)", fontweight="bold")
    ax.set_xlabel("Discount (%)")
    if ax == axes[0]:
        ax.set_ylabel("Profit (USD)")

plt.suptitle("Chart 9 — Discount vs Profit Scatter by Category",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_09_discount_category.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("   Saved: chart_09_discount_category.png")

# ── CHART 10: Sub-Category Discount vs Margin Bubble Chart ───────────────────
print("[Chart 10] Sub-Category Bubble Chart...")

sub_agg = df.groupby(["category", "sub_category"]).agg(
    avg_disc   = ("discount", "mean"),
    margin_pct = ("profit",   lambda x: x.sum() / df.loc[x.index, "sales"].sum() * 100),
    total_sales= ("sales",    "sum"),
).reset_index()

CAT_COL = {"Technology": BLUE, "Furniture": RED, "Office Supplies": GREEN}

fig, ax = plt.subplots(figsize=(11, 7))
for _, row in sub_agg.iterrows():
    color  = CAT_COL.get(row["category"], GRAY)
    size   = row["total_sales"] / 8000
    marker = "o" if row["margin_pct"] >= 0 else "X"
    ax.scatter(row["avg_disc"]*100, row["margin_pct"],
               s=size, c=color, alpha=0.7, marker=marker,
               edgecolors="white", linewidth=0.5)
    ax.annotate(row["sub_category"],
                (row["avg_disc"]*100, row["margin_pct"]),
                textcoords="offset points", xytext=(5, 3),
                fontsize=8, color="black")

ax.axhline(0,  color="black", linewidth=1,   linestyle="--")
ax.axvline(20, color=ORANGE,  linewidth=1.2, linestyle=":", label="20% disc threshold")

# Legend category
for cat, col in CAT_COL.items():
    ax.scatter([], [], c=col, s=80, label=cat, alpha=0.7, edgecolors="white", linewidth=0.5)

ax.set_xlabel("Average Discount (%)")
ax.set_ylabel("Profit Margin (%)")
ax.set_title("Chart 10 — Sub-Category: Avg Discount vs Profit Margin\n"
             "(Bubble size = Revenue | X marker = Loss-making)",
             fontsize=13, fontweight="bold", pad=12)
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_10_subcategory_bubble.png"), dpi=150)
plt.close()
print("   Saved: chart_10_subcategory_bubble.png")

# ── KEY INSIGHTS ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("KEY INSIGHTS - DISCOUNT ANALYSIS")
print("=" * 60)

no_disc  = df[df["discount"] == 0]
low_disc = df[(df["discount"] > 0) & (df["discount"] <= 0.20)]
high_disc= df[df["discount"] > 0.40]

print(f"""
Insight 1 — No Discount (0%):
  Orders : {len(no_disc):,}
  Margin : {no_disc['profit'].sum()/no_disc['sales'].sum()*100:.1f}%
  Avg Profit/Order: ${no_disc['profit'].mean():.1f}

Insight 2 — Low Discount (1-20%):
  Orders : {len(low_disc):,}
  Margin : {low_disc['profit'].sum()/low_disc['sales'].sum()*100:.1f}%
  Avg Profit/Order: ${low_disc['profit'].mean():.1f}

Insight 3 — High Discount (>40%):
  Orders : {len(high_disc):,}
  Margin : {high_disc['profit'].sum()/high_disc['sales'].sum()*100:.1f}%
  Avg Profit/Order: ${high_disc['profit'].mean():.1f}
  Loss Rate: {(high_disc['profit'] < 0).mean()*100:.0f}%

Insight 4 — Loss-making sub-categories:""")

loss_sub = sub_agg[sub_agg["margin_pct"] < 0].sort_values("margin_pct")
for _, r in loss_sub.iterrows():
    print(f"  -> {r['sub_category']:20s} margin: {r['margin_pct']:.1f}%  "
          f"avg_disc: {r['avg_disc']*100:.0f}%")

print(f"""
Recommendation:
  Cap discount at 20% for all products.
  For Furniture Tables & Bookcases: remove discount entirely.
  Estimated profit recovery: significant improvement in margin.
""")
print(f" Charts saved to: {CHART_DIR}")
print("\n" + "=" * 60)
print("HOAN THANH Notebook 02!")
print("Buoc tiep theo: 02_Python/03_RFM_Segmentation.py")
print("=" * 60)



GLOBAL SUPERSTORE - DISCOUNT IMPACT ANALYSIS

Data loaded: 51,290 rows

── DISCOUNT IMPACT SUMMARY ──────────────────────────────────
disc_bracket  orders  total_sales  total_profit  loss_orders  margin_pct  loss_rate_pct
0% (No disc)   29009 6.992411e+06  1.770695e+06            0   25.323101       0.000000
       1-10%    4679 1.962619e+06  3.381893e+05          901   17.231530      19.256251
      11-20%    6274 1.757261e+06  1.732548e+05         1463    9.859367      23.318457
      21-30%     967 3.825547e+05 -2.115561e+04          601   -5.530089      62.150982
      31-40%    3400 7.013431e+05 -1.661154e+05         2727  -23.685332      80.205882
        >40%    6961 8.463130e+05 -6.274110e+05         6852  -74.134634      98.434133

[Chart 7] Profit Margin by Discount Bracket...
   Saved: chart_07_discount_margin.png
[Chart 8] Loss Rate by Discount Bracket...
   Saved: chart_08_loss_rate.png
[Chart 9] Discount vs Profit by Category...
   Saved: chart_09_discount_category.png
[C